# Day 3 — Phase 1.1: Matrix Operations in NumPy
> **Date:** 2026-04-07  
> **Topic:** The NumPy patterns that power every AI framework — broadcasting, batch ops, `einsum`, and how PyTorch/TensorFlow use the same ideas

---

## Why This Day Matters

Days 1 and 2 gave you the *math*. Today you learn the *mechanics*: how to write that math efficiently in code.  
Every neural network forward pass, loss computation, and gradient update is secretly a sequence of these NumPy patterns.

| Pattern | Where it appears in AI |
|---|---|
| **Broadcasting** | Add bias `b` to every row of `W @ X` |
| **Batch matrix multiply** | Process 32 samples simultaneously |
| **`einsum`** | Attention scores, tensor contractions |
| **`reshape` / `transpose`** | Prepare data for conv layers, multi-head attention |
| **Vectorized ops vs loops** | 100–1000× speed difference |

---

## Sections

1. Broadcasting — the magic rule
2. Batch matrix operations
3. `reshape`, `transpose`, `squeeze`, `unsqueeze`
4. `einsum` — the universal tensor operation
5. Vectorized vs loop speed benchmarks
6. Putting it together: a neural network forward pass

---
## 1. Broadcasting — The Magic Rule

### Purpose
Broadcasting lets you do arithmetic between arrays of **different shapes** without copying data.  
It is used in *every single line* of AI code — adding biases, normalizing batches, computing losses.

---

### The Rule (3 steps)

When two arrays have different shapes, NumPy aligns them from the **right** and applies these rules:

1. If shapes have different number of dimensions → **prepend 1s** to the shorter shape
2. Dimensions of size **1** are stretched to match the other array
3. If shapes are incompatible (not 1 and not equal) → **error**

```
Examples:
  (3, 4)  +  (4,)    →  (4,) becomes (1,4) becomes (3,4)  ✅
  (3, 4)  +  (3, 1)  →  (3,1) becomes (3,4)               ✅
  (3, 4)  +  (3,)    →  (3,) becomes (1,3)... mismatch     ❌
```

**AI context:** A mini-batch output has shape `(batch, features)`.  
The bias vector has shape `(features,)`. Broadcasting adds it to every sample automatically.

In [1]:
import numpy as np

# ── Example 1: Add bias to a batch output ───────────────────
# Batch of 4 samples, each with 3 features
output = np.array([[1.0, 2.0, 3.0],
                   [4.0, 5.0, 6.0],
                   [7.0, 8.0, 9.0],
                   [0.0, 1.0, 2.0]])  # shape (4, 3)

bias = np.array([0.1, 0.2, 0.3])     # shape (3,)

result = output + bias               # (4,3) + (3,) → broadcast → (4,3)
print("output shape:", output.shape)
print("bias shape:  ", bias.shape)
print("result shape:", result.shape)
print("result:\n", result)

# ── Example 2: Subtract mean per feature (batch normalization idea) ──
mean = output.mean(axis=0)           # shape (3,) — mean of each column
std  = output.std(axis=0)            # shape (3,)
normalized = (output - mean) / std   # all broadcast!
print("\nNormalized (mean~0, std~1 per feature):")
print(np.round(normalized, 3))
print("Feature means after norm:", np.round(normalized.mean(axis=0), 10))

# ── Example 3: Outer product via broadcasting ────────────────
a = np.array([1, 2, 3])           # shape (3,)
b = np.array([10, 20, 30, 40])    # shape (4,)

outer = a[:, np.newaxis] * b[np.newaxis, :]  # (3,1) * (1,4) → (3,4)
print("\nOuter product (3,1) * (1,4) → (3,4):")
print(outer)
print("Same as np.outer:", np.allclose(outer, np.outer(a, b)))

output shape: (4, 3)
bias shape:   (3,)
result shape: (4, 3)
result:
 [[1.1 2.2 3.3]
 [4.1 5.2 6.3]
 [7.1 8.2 9.3]
 [0.1 1.2 2.3]]

Normalized (mean~0, std~1 per feature):
[[-0.73  -0.73  -0.73 ]
 [ 0.365  0.365  0.365]
 [ 1.461  1.461  1.461]
 [-1.095 -1.095 -1.095]]
Feature means after norm: [0. 0. 0.]

Outer product (3,1) * (1,4) → (3,4):
[[ 10  20  30  40]
 [ 20  40  60  80]
 [ 30  60  90 120]]
Same as np.outer: True


---
## 2. Batch Matrix Operations

### Purpose
In AI, we almost never process one sample at a time. We process a **batch** of 32, 128, or 512 samples simultaneously.  
NumPy and PyTorch handle this by treating the first dimension as the batch dimension.

---

### Shapes to Know

| Symbol | Meaning | Typical size |
|---|---|---|
| `B` | Batch size | 32–512 |
| `d_in` | Input features | 768 (BERT), 4096 (LLaMA) |
| `d_out` | Output features | same or different |
| `seq` | Sequence length (for text) | 512, 2048 |

```
Forward pass for one batch:
  X:  (B, d_in)    ← batch of input vectors
  W:  (d_in, d_out) ← weight matrix — shared across all samples!
  b:  (d_out,)     ← bias
  
  Y = X @ W + b    → (B, d_out)   ← one output per sample
```

**Key insight:** `X @ W` does B separate matrix-vector multiplications at once — no Python loop needed.

In [2]:
import numpy as np

np.random.seed(0)
B, d_in, d_out = 32, 64, 16

# ── One linear layer: Y = X @ W + b ─────────────────────────
X = np.random.randn(B, d_in)          # (32, 64) — batch of 32 samples
W = np.random.randn(d_in, d_out)      # (64, 16) — weight matrix
b = np.zeros(d_out)                    # (16,)    — bias

Y = X @ W + b                          # (32, 16) — batch of outputs
print(f"X shape: {X.shape}")
print(f"W shape: {W.shape}")
print(f"Y = X @ W + b  →  shape: {Y.shape}")

# ── Two-layer network ────────────────────────────────────────
W1 = np.random.randn(d_in, 32);  b1 = np.zeros(32)
W2 = np.random.randn(32, d_out); b2 = np.zeros(d_out)

H = np.maximum(0, X @ W1 + b1)        # ReLU activation: (32, 32)
Y2 = H @ W2 + b2                       # (32, 16)
print(f"\nTwo-layer network:")
print(f"  Layer 1: {X.shape} @ {W1.shape} → {H.shape}  (after ReLU)")
print(f"  Layer 2: {H.shape} @ {W2.shape} → {Y2.shape}")

# ── Batched 3D matrix multiply ───────────────────────────────
# In transformers: (batch, seq, d_model) @ (d_model, d_k) at once
seq = 10
X_seq = np.random.randn(B, seq, d_in)  # (32, 10, 64) — batch of sequences
W_q   = np.random.randn(d_in, 8)       # (64, 8)  — query projection

Q = X_seq @ W_q                         # (32, 10, 8) — query matrix for each sample
print(f"\nTransformer query projection:")
print(f"  X_seq {X_seq.shape} @ W_q {W_q.shape} → Q {Q.shape}")
print("  (batch=32, seq_len=10, d_k=8)")

X shape: (32, 64)
W shape: (64, 16)
Y = X @ W + b  →  shape: (32, 16)

Two-layer network:
  Layer 1: (32, 64) @ (64, 32) → (32, 32)  (after ReLU)
  Layer 2: (32, 32) @ (32, 16) → (32, 16)

Transformer query projection:
  X_seq (32, 10, 64) @ W_q (64, 8) → Q (32, 10, 8)
  (batch=32, seq_len=10, d_k=8)


---
## 3. `reshape`, `transpose`, `squeeze`, `unsqueeze`

### Purpose
Shape manipulation is *the most common source of bugs* in AI code. Understanding what these operations do (and don't do) to the underlying data is critical.

---

### Key Operations

| Operation | What it does | When to use |
|---|---|---|
| `reshape(new_shape)` | Reinterpret the data layout | Flatten image `(C,H,W) → (C*H*W,)` |
| `transpose(axes)` | Permute dimension order | Swap `(B, seq, d) → (seq, B, d)` |
| `T` | 2D shorthand for transpose | Flip `(m,n) → (n,m)` |
| `squeeze()` | Remove dimensions of size 1 | `(1, 8, 1) → (8,)` |
| `expand_dims` / `[:, np.newaxis]` | Add a size-1 dimension | `(8,) → (8,1)` for broadcasting |

**Important:** `reshape` and `transpose` do **not** copy data by default — they return a *view*.  
This is why shape bugs are subtle: you might change data through one variable and break another.

In [3]:
import numpy as np

# ── reshape ──────────────────────────────────────────────────
X = np.arange(24)                     # [0, 1, ..., 23]
print("Original:", X.shape)           # (24,)

A = X.reshape(4, 6)                   # 4 rows, 6 cols
B = X.reshape(2, 3, 4)                # 3D: 2 groups of 3×4
C = X.reshape(6, -1)                  # -1 = infer → (6, 4)
print(f"reshape(4,6): {A.shape}")
print(f"reshape(2,3,4): {B.shape}")
print(f"reshape(6,-1): {C.shape}  ← -1 auto-inferred")

# ── reshape is a VIEW — careful! ─────────────────────────────
X2 = np.array([[1, 2, 3], [4, 5, 6]])
flat = X2.reshape(-1)                 # view of X2
flat[0] = 99                          # modifies BOTH!
print(f"\nAfter flat[0]=99: X2[0,0] = {X2[0,0]}  ← view, not copy!")

# ── transpose ────────────────────────────────────────────────
M = np.random.randn(3, 4, 5)         # (batch=3, seq=4, features=5)
Mt = M.transpose(1, 0, 2)            # (seq=4, batch=3, features=5)
Mt2 = M.transpose(0, 2, 1)           # (batch=3, features=5, seq=4)
print(f"\nOriginal: {M.shape}")
print(f"transpose(1,0,2): {Mt.shape}  ← swap batch and seq")
print(f"transpose(0,2,1): {Mt2.shape}  ← swap seq and features")

# ── Real use case: prepare for matmul ─────────────────────────
# Attention: Q @ K.T per batch
# Q shape: (B, seq, d_k) — K.T needs (B, d_k, seq)
Q = np.random.randn(8, 10, 16)       # (B=8, seq=10, d_k=16)
K = np.random.randn(8, 10, 16)       # (B=8, seq=10, d_k=16)

scores = Q @ K.transpose(0, 2, 1)    # (8,10,16) @ (8,16,10) → (8,10,10)
print(f"\nAttention scores: Q {Q.shape} @ K.T {K.transpose(0,2,1).shape} → {scores.shape}")
print("  = (batch, seq, seq) attention score matrix")

# ── squeeze and expand_dims ───────────────────────────────────
v = np.array([1, 2, 3])              # (3,)
v_col = v[:, np.newaxis]             # (3, 1) — column vector
v_row = v[np.newaxis, :]             # (1, 3) — row vector
print(f"\nv: {v.shape}  → v_col: {v_col.shape}  v_row: {v_row.shape}")

squeezed = np.zeros((1, 5, 1))
print(f"squeeze((1,5,1)): {np.squeeze(squeezed).shape}")

Original: (24,)
reshape(4,6): (4, 6)
reshape(2,3,4): (2, 3, 4)
reshape(6,-1): (6, 4)  ← -1 auto-inferred

After flat[0]=99: X2[0,0] = 99  ← view, not copy!

Original: (3, 4, 5)
transpose(1,0,2): (4, 3, 5)  ← swap batch and seq
transpose(0,2,1): (3, 5, 4)  ← swap seq and features

Attention scores: Q (8, 10, 16) @ K.T (8, 16, 10) → (8, 10, 10)
  = (batch, seq, seq) attention score matrix

v: (3,)  → v_col: (3, 1)  v_row: (1, 3)
squeeze((1,5,1)): (5,)


---
## 4. `einsum` — The Universal Tensor Operation

### Purpose
`einsum` (Einstein summation) is a compact notation for **any tensor operation** — dot products, matrix multiplies, outer products, transposes, traces, and attention scores — all in one function.  
Once you learn to read `einsum`, the source code of transformer models becomes transparent.

---

### How to Read `einsum`

```
np.einsum('ij,jk->ik', A, B)
            ↑  ↑  ↑
            |  |  output indices (kept)
            |  second array indices
            first array indices
```

**Rule:** indices that appear in *inputs but not in output* are **summed over** (contracted).

| `einsum` string | Operation | NumPy equivalent |
|---|---|---|
| `'i,i->'` | Dot product | `np.dot(a, b)` |
| `'ij,jk->ik'` | Matrix multiply | `A @ B` |
| `'ij->ji'` | Transpose | `A.T` |
| `'i,j->ij'` | Outer product | `np.outer(a,b)` |
| `'ii->'` | Trace | `np.trace(A)` |
| `'bij,bjk->bik'` | Batch matmul | `A @ B` (batched) |
| `'bqd,bkd->bqk'` | Attention scores | `Q @ K.T` (per batch) |

In [4]:
import numpy as np

np.random.seed(1)

a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])
A = np.random.randn(3, 4)
B = np.random.randn(4, 5)

# ── Basic operations ─────────────────────────────────────────
print("=== Basic einsum ===")
print(f"Dot product     'i,i->':    {np.einsum('i,i->', a, b):.2f}  == {np.dot(a,b):.2f}")
print(f"Outer product   'i,j->ij':  shape {np.einsum('i,j->ij', a, b).shape}  == {np.outer(a,b).shape}")
print(f"Matrix multiply 'ij,jk->ik': shape {np.einsum('ij,jk->ik', A, B).shape}  == {(A@B).shape}")
print(f"Transpose       'ij->ji':    shape {np.einsum('ij->ji', A).shape}  == {A.T.shape}")
print(f"Trace           'ii->':      {np.einsum('ii->', np.eye(4)):.0f}  == {np.trace(np.eye(4)):.0f}")

# ── Batch matrix multiply ────────────────────────────────────
print("\n=== Batch matmul ===")
X3 = np.random.randn(8, 10, 64)   # (batch, seq, d_model)
W3 = np.random.randn(64, 16)       # (d_model, d_k)

# Without einsum:
out1 = X3 @ W3                     # (8, 10, 16) — works via broadcasting
# With einsum:
out2 = np.einsum('bsd,dk->bsk', X3, W3)  # b=batch, s=seq, d=model, k=output
print(f"X3 {X3.shape} @ W3 {W3.shape} → {out1.shape}")
print(f"einsum 'bsd,dk->bsk': {out2.shape}  equal: {np.allclose(out1, out2)}")

# ── Attention scores (transformer self-attention) ─────────────
print("\n=== Attention Scores ===")
B_a, seq, d_k = 4, 12, 8
Q = np.random.randn(B_a, seq, d_k)   # (batch, query_len, d_k)
K = np.random.randn(B_a, seq, d_k)   # (batch, key_len,   d_k)

# Q @ K.T per batch: for each batch, each query attends to all keys
scores_manual = Q @ K.transpose(0, 2, 1)            # (4, 12, 12)
scores_einsum = np.einsum('bqd,bkd->bqk', Q, K)     # same result
print(f"Q {Q.shape}, K {K.shape}")
print(f"Attention scores: {scores_einsum.shape}  (each of 4 batches: 12×12 query-key matrix)")
print(f"einsum == manual: {np.allclose(scores_manual, scores_einsum)}")

# ── Sum over axes ────────────────────────────────────────────
print("\n=== Axis summation ===")
C = np.random.randn(3, 4, 5)
print(f"Sum all:          'ijk->':    {np.einsum('ijk->', C).shape} scalar")
print(f"Sum over k:       'ijk->ij':  {np.einsum('ijk->ij', C).shape}")
print(f"Sum over i,k:     'ijk->j':   {np.einsum('ijk->j', C).shape}")

=== Basic einsum ===
Dot product     'i,i->':    32.00  == 32.00
Outer product   'i,j->ij':  shape (3, 3)  == (3, 3)
Matrix multiply 'ij,jk->ik': shape (3, 5)  == (3, 5)
Transpose       'ij->ji':    shape (4, 3)  == (4, 3)
Trace           'ii->':      4  == 4

=== Batch matmul ===
X3 (8, 10, 64) @ W3 (64, 16) → (8, 10, 16)
einsum 'bsd,dk->bsk': (8, 10, 16)  equal: True

=== Attention Scores ===
Q (4, 12, 8), K (4, 12, 8)
Attention scores: (4, 12, 12)  (each of 4 batches: 12×12 query-key matrix)
einsum == manual: True

=== Axis summation ===
Sum all:          'ijk->':    () scalar
Sum over k:       'ijk->ij':  (3, 4)
Sum over i,k:     'ijk->j':   (4,)


---
## 5. Vectorized Operations vs Loops

### Purpose
Python loops are **slow** — each iteration has interpreter overhead. NumPy pushes the loop down into C/Fortran where it runs 100–1000× faster.  
In AI training on large datasets, this difference is the gap between "runs in 2 minutes" and "runs overnight".

---

### The Golden Rule

> **Never loop over samples or features if a vectorized form exists.**

| ❌ Slow (Python loop) | ✅ Fast (vectorized) |
|---|---|
| `for x in X: y = w @ x + b` | `Y = X @ W + b` |
| `for i in range(n): loss += (y[i]-t[i])**2` | `loss = np.sum((y-t)**2)` |
| `for i in range(n): norm[i] = np.sum(X[i]**2)` | `norm = np.sum(X**2, axis=1)` |

In [5]:
import numpy as np
import time

np.random.seed(0)
N, D = 10_000, 512
X = np.random.randn(N, D)
w = np.random.randn(D)
b = 0.5

# ── Dot product per sample: loop vs vectorized ───────────────
# Slow: Python loop
t0 = time.perf_counter()
y_loop = np.zeros(N)
for i in range(N):
    y_loop[i] = np.dot(X[i], w) + b
t_loop = time.perf_counter() - t0

# Fast: vectorized
t0 = time.perf_counter()
y_vec = X @ w + b
t_vec = time.perf_counter() - t0

print(f"Loop:       {t_loop*1000:.2f} ms")
print(f"Vectorized: {t_vec*1000:.2f} ms")
print(f"Speedup:    {t_loop/t_vec:.0f}×")
print(f"Results equal: {np.allclose(y_loop, y_vec)}")

# ── MSE loss: loop vs vectorized ────────────────────────────
targets = np.random.randn(N)

t0 = time.perf_counter()
loss_loop = sum((y_vec[i] - targets[i])**2 for i in range(N)) / N
t_loss_loop = time.perf_counter() - t0

t0 = time.perf_counter()
loss_vec = np.mean((y_vec - targets)**2)
t_loss_vec = time.perf_counter() - t0

print(f"\nMSE loss:")
print(f"  Loop:       {t_loss_loop*1000:.2f} ms  → {loss_loop:.4f}")
print(f"  Vectorized: {t_loss_vec*1000:.2f} ms  → {loss_vec:.4f}")
print(f"  Speedup:    {t_loss_loop/max(t_loss_vec,1e-9):.0f}×")

# ── Row norms: loop vs vectorized ────────────────────────────
t0 = time.perf_counter()
norms_loop = np.array([np.sqrt(np.sum(X[i]**2)) for i in range(N)])
t_norm_loop = time.perf_counter() - t0

t0 = time.perf_counter()
norms_vec = np.linalg.norm(X, axis=1)
t_norm_vec = time.perf_counter() - t0

print(f"\nRow norms:")
print(f"  Loop:       {t_norm_loop*1000:.2f} ms")
print(f"  Vectorized: {t_norm_vec*1000:.2f} ms")
print(f"  Speedup:    {t_norm_loop/max(t_norm_vec,1e-9):.0f}×")

Loop:       13.37 ms
Vectorized: 4.04 ms
Speedup:    3×
Results equal: True

MSE loss:
  Loop:       12.23 ms  → 586.7986
  Vectorized: 0.28 ms  → 586.7986
  Speedup:    43×

Row norms:
  Loop:       120.94 ms
  Vectorized: 13.75 ms
  Speedup:    9×


---
## 6. Putting It Together: A Neural Network Forward Pass

### Purpose
Combine everything above to build a complete 3-layer neural network forward pass using only NumPy.  
This is **exactly** what PyTorch does internally — just with autograd on top.

---

### Architecture

```
Input X:  (B, 784)       ← e.g. flattened 28×28 images (MNIST-style)
  ↓  W1: (784, 128) + b1
Hidden H1: (B, 128)  → ReLU
  ↓  W2: (128, 64)  + b2
Hidden H2: (B, 64)   → ReLU
  ↓  W3: (64, 10)   + b3
Logits:   (B, 10)        ← 10 class scores
  ↓  Softmax
Probs:    (B, 10)        ← probabilities (sum to 1 per sample)
  ↓  Cross-entropy loss
Loss:     scalar
```

In [6]:
import numpy as np

np.random.seed(42)

# ── Hyperparameters ──────────────────────────────────────────
B       = 32      # batch size
d_in    = 784     # input features (e.g. 28×28 image)
d_h1    = 128
d_h2    = 64
d_out   = 10      # 10 classes

# ── Xavier initialization (controls eigenvalue spread) ──────
def xavier(fan_in, fan_out):
    """Xavier/Glorot uniform initialization."""
    limit = np.sqrt(6.0 / (fan_in + fan_out))
    return np.random.uniform(-limit, limit, (fan_in, fan_out))

W1 = xavier(d_in, d_h1);  b1 = np.zeros(d_h1)
W2 = xavier(d_h1, d_h2);  b2 = np.zeros(d_h2)
W3 = xavier(d_h2, d_out);  b3 = np.zeros(d_out)

# ── Activation functions ─────────────────────────────────────
def relu(x): return np.maximum(0, x)

def softmax(x):
    # Numerically stable: subtract max before exp
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

def cross_entropy(probs, labels):
    """labels: integer class indices, shape (B,)"""
    n = len(labels)
    log_p = np.log(probs[np.arange(n), labels] + 1e-9)
    return -log_p.mean()

# ── Forward pass ─────────────────────────────────────────────
X  = np.random.randn(B, d_in)             # fake input batch
y  = np.random.randint(0, d_out, size=B)  # fake labels

H1    = relu(X  @ W1 + b1)   # (B, 128)
H2    = relu(H1 @ W2 + b2)   # (B, 64)
logits = H2 @ W3 + b3         # (B, 10)
probs  = softmax(logits)       # (B, 10) — probabilities
loss   = cross_entropy(probs, y)

print("=== Forward Pass ===")
print(f"X:      {X.shape}")
print(f"H1:     {H1.shape}   (after ReLU)")
print(f"H2:     {H2.shape}    (after ReLU)")
print(f"Logits: {logits.shape}")
print(f"Probs:  {probs.shape}  (each row sums to: {probs.sum(axis=1)[:3].round(6)})")
print(f"Loss:   {loss:.4f}")
print(f"\nPredicted classes: {probs.argmax(axis=1)[:8]}")
print(f"True classes:      {y[:8]}")
print(f"Accuracy on batch: {(probs.argmax(axis=1) == y).mean()*100:.1f}%  (random ≈ 10%)")

# ── Shape audit — verify every step ─────────────────────────
print("\n=== Shape Audit ===")
steps = [
    ("X @ W1",         X @ W1,              f"({B},{d_in}) @ ({d_in},{d_h1})"),
    ("+b1 broadcast",  X @ W1 + b1,         f"({B},{d_h1}) + ({d_h1},)"),
    ("H1 @ W2",        H1 @ W2,             f"({B},{d_h1}) @ ({d_h1},{d_h2})"),
    ("H2 @ W3",        H2 @ W3,             f"({B},{d_h2}) @ ({d_h2},{d_out})"),
]
for name, arr, desc in steps:
    print(f"  {name:<20} {desc:<30} → {arr.shape}")

=== Forward Pass ===
X:      (32, 784)
H1:     (32, 128)   (after ReLU)
H2:     (32, 64)    (after ReLU)
Logits: (32, 10)
Probs:  (32, 10)  (each row sums to: [1. 1. 1.])
Loss:   2.9184

Predicted classes: [4 4 8 4 4 0 4 0]
True classes:      [1 8 9 0 3 5 7 5]
Accuracy on batch: 9.4%  (random ≈ 10%)

=== Shape Audit ===
  X @ W1               (32,784) @ (784,128)           → (32, 128)
  +b1 broadcast        (32,128) + (128,)              → (32, 128)
  H1 @ W2              (32,128) @ (128,64)            → (32, 64)
  H2 @ W3              (32,64) @ (64,10)              → (32, 10)


---
## 7. Why This Matters for AI

| Pattern learned today | Where it appears |
|---|---|
| `X @ W + b` with broadcasting | Every linear layer in every neural network |
| Batch dimension (B, ...) | Every forward pass processes entire batches |
| 3D matmul `(B, seq, d) @ (d, k)` | Transformer Q, K, V projections |
| `einsum 'bqd,bkd->bqk'` | Attention score matrix |
| `transpose(0,2,1)` | K.T for attention, Conv feature maps |
| Xavier init via `reshape` / `sqrt` | Controlling gradient flow via eigenvalue spread |
| `softmax` with `keepdims=True` broadcast | Classification head, attention weights |

**Key insight:** Every framework (PyTorch, JAX, TensorFlow) is just a library of these patterns with automatic differentiation on top. If you understand the shapes and operations here, you can read and debug any model's source code.

---
## Exercise 3

**Questions:**
1. Why does `(3,4) + (4,)` work but `(3,4) + (3,)` fail in NumPy?
2. In the 3-layer forward pass, what is the shape at each step if batch size is 64 instead of 32?
3. Rewrite `Q @ K.transpose(0,2,1)` using `einsum`. What string notation would you use?
4. The `softmax` function uses `x - x.max(...)`. Why is this subtraction necessary? What would go wrong without it?

---
## Summary

| Concept | Syntax | AI Usage |
|---|---|---|
| Broadcasting | `output + bias` (shapes auto-match) | Add bias to all samples |
| Batch matmul | `X @ W` where X is (B, d) | Linear layer for a whole batch |
| Reshape | `x.reshape(B, -1)` | Flatten images, merge dimensions |
| Transpose | `K.transpose(0,2,1)` | Attention, conv layer prep |
| einsum | `'bqd,bkd->bqk'` | Attention scores, tensor contractions |
| Vectorize | Avoid Python loops over samples | 100–1000× speedup |

---
## Next: Day 4 — Derivatives & Calculus
The math behind **backpropagation**: partial derivatives, chain rule, and gradient descent — how a network learns from its mistakes.

---
## My Notes
> *(Write your own observations, questions, and breakthroughs here)*